In [1]:
# !pip install codecarbon
# !pip install -e .

import pandas as pd
import pickle
from trust_library.core import TrustEvaluator
from trust_library.factsheet import (
    load_factsheet_default,
    load_factsheet,
    save_factsheet, 
    create_factsheet_interactive,
    create_factsheet,
)
from trust_library.fairness import statistical_parity_difference

import matplotlib.pyplot as plt
import seaborn as sns
import json

from trust_library.utils import to_json_safe
import plotly.express as px

In [2]:
# factsheet = create_factsheet_interactive()
# print(load_factsheet_default()) 
# fs = create_factsheet({
# "general": {
#     "target_column": "Target"
# },
# "fairness": {
#     "protected_feature": "Group",
#     "protected_values": [1],      # Valor que identifica al grupo desprotegido
#     "favorable_outcomes": [1]     # Valor que identifica el éxito
# }
# })
# save_factsheet(fs, "factsheet.json")

factsheet = load_factsheet_default() #load_factsheet("factsheet.json")

with open("model_tree.pkl", "rb") as f:
    loaded_model = pickle.load(f)

train_loaded = pd.read_csv("train.csv")
test_loaded = pd.read_csv("test.csv")

evaluator = TrustEvaluator(
    model=loaded_model,
    train_data=train_loaded,
    test_data=test_loaded,
    factsheet=factsheet
)

In [3]:
# results = evaluator.evaluate()

# print("\n=== RESULTADOS FINAL ===")
# print(f"Global Trust Score: {results['trust_score']}")
# print(f"Pillar Scores:\n{json.dumps(results['pillar_score'], indent=4)}")
# print("\nDetalles (Scores brutos):\n" + json.dumps(results['details'], indent=4))

# print("\nPropiedades calculadas:\n" + json.dumps(to_json_safe(results['properties']), indent=4))
# print("\n=== EXPLICACIÓN DEL SCORE ===")
# print(json.dumps(results['explanation'], indent=4))

In [4]:
# y_pred = loaded_model.predict(
#     test_loaded.drop(
#         columns=[factsheet["general"]["target_column"]["value"]]
#     )
# )

# group_mask = (
#     test_loaded[factsheet["fairness"]["protected_feature"]["value"]]
#     == factsheet["fairness"]["protected_values"]["value"][0]
# )

y_pred = loaded_model.predict(test_loaded.drop("Target", axis=1))
group_mask = test_loaded["Group"] == 1

statistical_parity_difference(y_pred,group_mask)

{'value': 0.37903622652710833,
 'favored_ratio_protected': 0.5891047998671317,
 'favored_ratio_unprotected': 0.2100685733400234,
 'n_protected': 6021,
 'n_unprotected': 5979,
 'n_protected_favored': 3547,
 'n_unprotected_favored': 1256}

In [5]:
subset = evaluator.evaluate(["explainability"], show_nan=True)
# Imprimimos subset formateado
print("\n=== SUBSET DE PILLARS (Fairness, Privacy, Accountability y Sustainability) ===")
print(json.dumps(subset, indent=4))

Computing Explainability metrics...
Metric 'sparsity' computed in 0.00 seconds.
Metric 'feature_entropy' computed in 0.00 seconds.
Metric 'topk_concentration' computed in 0.00 seconds.
Metric 'algorithm_class' computed in 0.00 seconds.
Metric 'correlated_features' computed in 0.00 seconds.
Metric 'model_size' computed in 0.00 seconds.
Metric 'feature_relevance' computed in 0.00 seconds.
Metric 'number_of_rules' computed in 0.00 seconds.
Metric 'average_rule_length' computed in 0.00 seconds.
Metric 'rule_stats' computed in 0.00 seconds.
Metric 'tree_depth' computed in 0.00 seconds.
Metric 'interaction_strength' computed in 0.00 seconds.
Metric 'faithfulness' computed in 0.00 seconds.
Metric 'monotonicity' computed in 0.00 seconds.
Metric 'infidelity' computed in 1.42 seconds.
Metric 'alpha_score' computed in 0.00 seconds.
Metric 'xai_ease_score' computed in 0.00 seconds.
Metric 'position_parity' computed in 0.00 seconds.
Metric 'rank_alignment' computed in 0.00 seconds.
Metric 'spread_r

In [ ]:
subset = evaluator.evaluate(["fairness", "explainability", "accountability", "sustainability"], show_nan=True)
# Imprimimos subset formateado
print("\n=== SUBSET DE PILLARS (Fairness, Privacy, Accountability y Sustainability) ===")
print(json.dumps(subset, indent=4))

In [ ]:
evaluator.plot_results()

In [ ]:
evaluator.plot_radar()

In [ ]:
# subset = evaluator.evaluate(["robustness"], show_nan=True)
# # Imprimimos subset formateado
# print("\n=== SUBSET DE PILLARS (Robustness) ===")
# print(json.dumps(subset, indent=4))

In [ ]:
with open("model_SVM.pkl", "rb") as f:
    loaded_model2 = pickle.load(f)
evaluator2 = TrustEvaluator(loaded_model2, train_loaded, test_loaded, factsheet)
subset2 = evaluator2.evaluate(["fairness", "accountability", "sustainability"], show_nan=True)



Computing Fairness metrics...
Computing Accountability metrics...
Computing Sustainability metrics...


In [ ]:

TrustEvaluator.compare_radar({
    "RandomForest": subset,
    "SVM": subset2
})

TrustEvaluator.compare_all_bars({
    "RandomForest": subset,
    "SVM": subset2
})